# AmsiNet - Step 15: validation-set comparison of the two word heads inside the fusion (CPU, about 2 minutes)
Reuses the Step 9 setup. Fits the fusion weights and tau* on validation exactly as Step 9, then reports the VALIDATION macro-F1 of the fused systems, so the choice of word head does not depend on the test set. Paste back the last two tables.

In [ ]:
# 1. Setup
from google.colab import drive
import os
if not os.path.ismount('/content/drive'): drive.mount('/content/drive')
assert os.path.exists('/content/drive/MyDrive/amsinet/data/bstd_index.json'), 'Drive is not mounted or amsinet/data is missing: reconnect and run this cell again'
import json, numpy as np, pandas as pd
from scipy.special import softmax
from scipy import stats
ROOT = '/content/drive/MyDrive/amsinet'
CFG = dict(SEEDS=[0, 1, 2, 3, 4], VARIANTS=['word', 'word_moe'], EPS=1e-4, KEEP=0.90)
D = json.load(open(f'{ROOT}/data/bstd_index.json', encoding='utf-8'))
SCRIPTS, INDEX = D['scripts'], D['index']; SIDX = {s: i for i, s in enumerate(SCRIPTS)}; C = len(SCRIPTS); IDX = {r['id']: r for r in INDEX}
META = pd.read_csv(f'{ROOT}/data/crops_meta.csv'); REGDF = pd.read_csv(f'{ROOT}/data/regions_meta.csv')
eff = pd.read_csv(f'{ROOT}/agents/agent_efficiency_cpu.csv').set_index('agent')['GFLOPs']
flat_df = pd.read_csv(f'{ROOT}/results/backbone_efficiency.csv'); FLAT = float(flat_df[flat_df.agent == 'Flat baseline']['GFLOPs'].iloc[0])
missing = [f'{t}_seed{s}' for s in CFG['SEEDS'] for t in ('block', 'line', 'word', 'word_moe') if not os.path.exists(f'{ROOT}/agents/{t}_seed{s}_test_probs.npy')]
print('missing agent files:', missing or 'none')

In [ ]:
# 2. Helpers (same definitions as Steps 6 and 8)
def load(tag, sp):
    P = np.load(f'{ROOT}/agents/{tag}_{sp}_probs.npy').astype(np.float32); pa = pd.read_csv(f'{ROOT}/agents/{tag}_{sp}_paths.csv'); return P, pa.path.tolist()
def region_lookup(level, tag, sp):
    P, paths = load(tag, sp); pos = {p: i for i, p in enumerate(paths)}
    d = REGDF[(REGDF.level == level) & (REGDF.split == sp)]
    return {(r.image_id, r.rid): P[pos[r.path]] for r in d.itertuples() if r.path in pos}
def table(sp, wtag, ltag, btag):
    d = META[META.split == sp].reset_index(drop=True); n = len(d)
    Pw, pw = load(wtag, sp); assert pw == d.path.tolist(), 'word probabilities are not aligned with crops_meta.csv'
    logP = np.log(np.clip(Pw, CFG['EPS'], 1)); lid, bid = [], []
    for iid, k in zip(d.image_id, d.word_idx):
        w = IDX[iid]['words'][k]; lid.append(w.get('line', -1)); bid.append(w.get('block', -1))
    img = d.image_id.values; lkey = img.astype(str) + '|l' + np.array(lid).astype(str); bkey = img.astype(str) + '|b' + np.array(bid).astype(str)
    lg, lu = pd.factorize(lkey); bg, bu = pd.factorize(bkey)
    def loo(g, ng):
        S = np.zeros((ng, C)); cnt = np.zeros(ng); np.add.at(S, g, logP); np.add.at(cnt, g, 1); o = cnt[g] - 1
        F = np.zeros((n, C)); m = o > 0; F[m] = (S[g][m] - logP[m]) / o[m][:, None]; return F
    def reg(level, tag, key):
        L = region_lookup(level, tag, sp); F = np.zeros((n, C))
        for i, (iid, kk) in enumerate(zip(img, key)):
            v = L.get((iid, f'{level[0]}{kk}'))
            if v is not None: F[i] = np.log(np.clip(v, CFG['EPS'], 1))
        return F
    ui, inv = np.unique(img, return_inverse=True); l_img = np.zeros(len(lu), int); l_img[lg] = inv; b_img = np.zeros(len(bu), int); b_img[bg] = inv
    return dict(y=np.array([SIDX[s] for s in d.script]), word=logP, line_ctx=loo(lg, len(lu)), block_ctx=loo(bg, len(bu)),
                line_agent=reg('line', ltag, lid), block_agent=reg('block', btag, bid),
                img=img, inv=inv, n_img=len(ui), lg=lg, bg=bg, nl=len(lu), nb=len(bu), l_img=l_img, b_img=b_img)
def confmat(y, p): return np.bincount(y * C + p, minlength=C * C).reshape(C, C)
def f1_from(cm):
    tp = np.diag(cm).astype(float); fp = cm.sum(0) - tp; fn = cm.sum(1) - tp; den = 2 * tp + fp + fn; return np.where(den > 0, 2 * tp / np.maximum(den, 1), 0.0)
def metr(y, p): cm = confmat(y, p); return dict(acc=100 * float(np.trace(cm) / cm.sum()), f1=100 * float(f1_from(cm).mean()))
def unc(s): p = softmax(s, axis=1); return -(p * np.log(np.clip(p, 1e-12, 1))).sum(1) / np.log(C)
KEYS = ['line_ctx', 'block_ctx', 'line_agent', 'block_agent']; GRID_W = [0, .1, .25, .5, 1, 1.5, 2, 3]
def cascade(T, tau, W):
    s = T['word'] + W['line_ctx'] * T['line_ctx'] + W['block_ctx'] * T['block_ctx']; u = unc(s)
    li = np.zeros(T['nl'], bool); np.logical_or.at(li, T['lg'], u > tau); s = s + W['line_agent'] * T['line_agent'] * li[T['lg']][:, None]
    u = unc(s); bi = np.zeros(T['nb'], bool); np.logical_or.at(bi, T['bg'], u > tau); s = s + W['block_agent'] * T['block_agent'] * bi[T['bg']][:, None]
    return s, li, bi
def cost(T, li, bi, GF):
    nw = np.bincount(T['inv'], minlength=T['n_img']).astype(float); nl = np.bincount(T['l_img'][li], minlength=T['n_img']).astype(float); nb = np.bincount(T['b_img'][bi], minlength=T['n_img']).astype(float)
    return float((nw * GF['word'] + nl * GF['line'] + nb * GF['block']).mean())
TAUS = [0, .02, .05, .1, .15, .2, .3, .4, .5, .6, .7, .8, .9, 1.01]

In [ ]:
rows = []
for variant in ['word', 'word_moe']:
    for seed in CFG['SEEDS']:
        wtag, ltag, btag = f'{variant}_seed{seed}', f'line_seed{seed}', f'block_seed{seed}'
        GF = dict(word=float(eff[variant]), line=float(eff['line']), block=float(eff['block']))
        TV = table('val', wtag, ltag, btag)
        def full(T, W): return T['word'] + sum(W[k] * T[k] for k in KEYS)
        vf1 = lambda W: metr(TV['y'], full(TV, W).argmax(1))['f1']
        W = {k: 0.0 for k in KEYS}
        for _ in range(2):
            for k in KEYS: W[k] = max(GRID_W, key=lambda g: vf1({**W, k: g}))
        cv = {t: (lambda r: (metr(TV['y'], r[0].argmax(1)), cost(TV, r[1], r[2], GF)))(cascade(TV, t, W)) for t in TAUS}
        f_full, f_ctx = cv[0][0]['f1'], cv[1.01][0]['f1']; target = f_ctx + CFG['KEEP'] * (f_full - f_ctx)
        ok = [t for t in TAUS if cv[t][0]['f1'] >= target]; tau = min(ok, key=lambda t: cv[t][1]) if ok else 0.0
        wo = metr(TV['y'], TV['word'].argmax(1))
        rows.append((variant, seed, tau, wo['f1'], cv[1.01][0]['f1'], cv[tau][0]['f1'], cv[0][0]['f1'], cv[tau][0]['acc'], cv[0][0]['acc']))
        print(variant, seed, 'tau*', tau, '| val macro-F1: word %.2f, +context %.2f, cascade %.2f, full %.2f' % rows[-1][3:7])
V = pd.DataFrame(rows, columns=['head', 'seed', 'tau', 'val F1 word', 'val F1 +context', 'val F1 cascade', 'val F1 full', 'val acc cascade', 'val acc full'])
def ci(v): v = np.asarray(v, float); return stats.t.ppf(.975, len(v) - 1) * v.std(ddof=1) / np.sqrt(len(v))
S = V.groupby('head').agg(['mean', ci]).drop(columns=['seed', 'tau']); S.columns = [f'{a} ({b})' for a, b in S.columns]; display(S.round(2))
piv = V.pivot(index='seed', columns='head', values='val F1 cascade'); d = (piv['word_moe'] - piv['word']).values
pivf = V.pivot(index='seed', columns='head', values='val F1 full'); df = (pivf['word_moe'] - pivf['word']).values
print(f'VALIDATION macro-F1, MoE minus linear: cascade {d.mean():+.2f} +/- {ci(d):.2f} | full fusion {df.mean():+.2f} +/- {ci(df):.2f} (positive = MoE better)')